# Python `str.join()`: Master Linear O(N) String Assembly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plus2net/Python-basics/blob/main/python_string_join_practice.ipynb)

Welcome to the interactive companion notebook for the **[plus2net Python str.join() Tutorial](https://www.plus2net.com/python/string-join.php)**.

### Learning Objectives:
1. Master the invocation paradigm of `separator.join(iterable)`.
2. Understand why repeated `+=` string concatenation is O(N^2) quadratic while `join()` is O(N) linear.
3. Join lists, tuples, sets, dictionary keys/values, and streaming generator expressions.
4. Safely handle non-string elements using `map(str, ...)` and comprehension conversions.
5. Build real-world text pipelines: CSV rows, Oxford comma sentences, and SQL query builders.
6. Validate implementations with automated self-checking unit assertions.

## 1. Basic Joining & The Separator-First Architecture

In Python, `join()` is invoked on the separator string rather than the collection.
This allows strings to concatenate any arbitrary iterable (including generators and custom iterators).

In [ ]:
words = ["Python", "is", "fast", "and", "clean"]

# 1. Basic join with space
sentence = " ".join(words)
print("Space join:    ", sentence)
assert sentence == "Python is fast and clean"

# 2. Join with hyphen
slug = "-".join(words)
print("Hyphen join:   ", slug)
assert slug == "Python-is-fast-and-clean"

# 3. Empty string separator (direct concatenation)
compact = "".join(["a", "b", "c"])
assert compact == "abc"

# 4. Boundary cases: empty iterable and singleton
assert ", ".join([]) == ""
assert ", ".join(["alone"]) == "alone"

print("\nAll basic joining assertions passed!")


## 2. Performance Microbenchmark: `str.join()` O(N) vs. Loop `+=` O(N^2)

Because Python strings are immutable, `result += item` allocates a new string and copies over all previous data on every single step.
`str.join()` allocates memory once.

In [ ]:
import time

elements = [f"token_{i}" for i in range(10000)]

# Benchmark 1: Repeated += loop
t0 = time.perf_counter()
loop_res = ""
for item in elements:
    loop_res += item + ", "
t_loop = time.perf_counter() - t0

# Benchmark 2: str.join()
t0 = time.perf_counter()
join_res = ", ".join(elements)
t_join = time.perf_counter() - t0

print(f"Loop += execution time : {t_loop:.5f}s")
print(f"str.join() execution time: {t_join:.5f}s")
print(f"str.join() speedup factor: {t_loop / t_join:.1f}x faster!")


## 3. Joining Diverse Iterables

Demonstrating `join()` with tuples, sets, dictionaries, and generator expressions.

In [ ]:
# Tuples
assert ":".join(("192", "168", "1", "1")) == "192:168:1:1"

# Dictionaries: joins keys by default; values explicitly
headers = {"Host": "plus2net.com", "Accept": "application/json"}
assert "; ".join(headers) == "Host; Accept"
assert "; ".join(headers.values()) == "plus2net.com; application/json"

# Generator expressions: zero memory overhead for intermediate lists
gen = (f"row_{x}" for x in range(3))
assert " | ".join(gen) == "row_0 | row_1 | row_2"

# Characters in a string
assert ".".join("DATA") == "D.A.T.A"

print("All iterable types assertions passed!")


## 4. Handling Non-String Elements & Type Conversion

Passing non-string elements directly raises a `TypeError`.

In [ ]:
numbers = [10, 20, 30, 40]

# 1. TypeError when passing integers directly
try:
    ", ".join(numbers)
    assert False, "Expected TypeError not raised"
except TypeError as e:
    assert "expected str instance" in str(e)

# 2. Conversion via map(str, ...)
mapped = ", ".join(map(str, numbers))
print("Via map():      ", mapped)
assert mapped == "10, 20, 30, 40"

# 3. Conversion via generator comprehension
gen_converted = ", ".join(str(n) for n in numbers)
print("Via generator:", gen_converted)
assert gen_converted == "10, 20, 30, 40"

print("\nType conversion assertions passed!")


## 5. Real-World Engineering Recipes

Explore three common software engineering patterns utilizing `str.join()`:
1. CSV Tabular Row Serializer with Escaping
2. Natural Language English List Formatter (Oxford Comma)
3. Dynamic SQL `IN` Clause Generator

In [ ]:
# Recipe 1: CSV Serializer
def serialize_csv_row(fields: list, delimiter: str = ",") -> str:
    escaped = []
    for f in fields:
        s = str(f)
        if delimiter in s or '"' in s or "\n" in s:
            inner = s.replace('"', '""')
            s = f'"{inner}"'
        escaped.append(s)
    return delimiter.join(escaped)

csv_out = serialize_csv_row(["Alice", "Engineer, Staff", 120000])
print("Recipe 1 CSV:", csv_out)
assert csv_out == 'Alice,"Engineer, Staff",120000'

# Recipe 2: Oxford comma formatter
def format_oxford(items: list[str]) -> str:
    if not items:
        return ""
    if len(items) == 1:
        return items[0]
    if len(items) == 2:
        return f"{items[0]} and {items[1]}"
    return f"{', '.join(items[:-1])}, and {items[-1]}"

print("Recipe 2 Oxford:", format_oxford(["Python", "SQL", "Docker"]))
assert format_oxford(["Python", "SQL", "Docker"]) == "Python, SQL, and Docker"

# Recipe 3: SQL IN clause
def build_sql_in(table: str, col: str, ids: list[int]) -> str:
    placeholders = ", ".join("?" for _ in ids)
    return f"SELECT * FROM {table} WHERE {col} IN ({placeholders})"

sql = build_sql_in("users", "user_id", [10, 20, 30])
print("Recipe 3 SQL:   ", sql)
assert sql == "SELECT * FROM users WHERE user_id IN (?, ?, ?)"

print("\nAll engineering recipes passed!")


## 6. Production Implementation & Self-Checking Unit Assertions

This production-grade `StringAssembler` class encapsulates mixed-sequence conversion, URL query formatting, and clean POSIX path assembly.

In [ ]:
class StringAssembler:
    @staticmethod
    def join_mixed_sequence(delimiter: str, items: list) -> str:
        """Safely converts all elements to string before joining."""
        return delimiter.join(str(x) for x in items)

    @staticmethod
    def build_query_string(params: dict[str, any]) -> str:
        """Encodes dictionary key-value pairs into URL query string."""
        return "&".join(f"{k}={v}" for k, v in params.items())

    @staticmethod
    def build_path(*segments: str) -> str:
        """Combines path segments cleanly without double slashes."""
        stripped = [seg.strip("/") for seg in segments if seg.strip("/")]
        return "/" + "/".join(stripped)


# ==========================================
# UNIT ASSERTION SUITE
# ==========================================
assembler = StringAssembler()

# 1. Mixed sequence conversion
assert assembler.join_mixed_sequence(", ", [1, "two", 3.0, True]) == "1, two, 3.0, True"

# 2. Query string building
query_data = {"q": "python", "page": 1, "sort": "asc"}
assert assembler.build_query_string(query_data) == "q=python&page=1&sort=asc"

# 3. Path assembly
assert assembler.build_path("api/", "/v1/", "users/list") == "/api/v1/users/list"

print("All str.join() production unit assertions passed successfully!")
